In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns 
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity
import pingouin as pg

# Очистка

In [2]:
df = pd.read_excel('anketa520047-2026-04-29 (valid_units).xlsx')

In [3]:
df = df.iloc[1:].reset_index(drop=True)

In [4]:
df['itime'].value_counts()

itime
28.04.2026    1585
27.04.2026       1
Name: count, dtype: int64

In [5]:
df['itime'] = pd.to_datetime(df['itime'], dayfirst=True, errors='coerce')

In [6]:
df = df[df['itime'] == '2026-04-28']

In [7]:
df

,status,invitation,relevance,lurker,recnum,code,testdata,itime,Q1,Q3a,...,date_9,date_10,date_11,Browser,BrowserVersion,OS,Device,Referer,Unsubscribed,Language
1,5,0,1,0,53,NaN,0,2026-04-28,1,5,...,28.04.2026 10:17:59,28.04.2026 10:18:15,NaN,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,Chrome 0.0,Win10,PC,https://iframe-tasks.yandex/,0,Russian
2,5,0,1,0,54,NaN,0,2026-04-28,1,5,...,28.04.2026 10:10:46,28.04.2026 10:11:05,NaN,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,Chrome 0.0,Win10,PC,https://iframe-tasks.yandex/,0,Russian
3,5,0,1,0,55,NaN,0,2026-04-28,1,5,...,28.04.2026 10:13:59,28.04.2026 10:14:12,NaN,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,Chrome 134.0,Win10,PC,https://iframe-tasks.yandex/,0,Russian
4,5,0,1,0,56,NaN,0,2026-04-28,1,5,...,28.04.2026 10:20:22,NaN,NaN,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,Chrome 0.0,Win10,PC,https://iframe-tasks.yandex/,0,Russian
5,5,0,1,0,57,NaN,0,2026-04-28,1,5,...,NaN,NaN,NaN,Mozilla/5.0 (Linux; arm_64; Android 11; RMX326...,Chrome 0.0,Android,Phone,0,0,Russian
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1581,6,0,1,0,2070,NaN,0,2026-04-28,1,5,...,28.04.2026 17:59:11,28.04.2026 17:59:34,28.04.2026 18:00:00,Mozilla/5.0 (Linux; Android 10; K) AppleWebKit...,Chrome 0.0,Android,Phone,0,0,NaN
1582,5,0,1,0,2071,NaN,0,2026-04-28,1,5,...,28.04.2026 18:15:31,28.04.2026 18:15:53,NaN,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,Chrome 0.0,Win10,PC,https://iframe-tasks.yandex/,0,NaN
1583,6,0,1,0,2072,NaN,0,2026-04-28,1,5,...,28.04.2026 18:28:01,28.04.2026 18:28:39,28.04.2026 18:28:58,Mozilla/5.0 (Linux; arm_64; Android 13; 23053R...,Chrome 0.0,Android,Phone,https://iframe-tasks.yandex/,0,NaN
1584,5,0,1,0,2073,NaN,0,2026-04-28,1,5,...,NaN,NaN,NaN,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,Chrome 0.0,Win10,PC,https://iframe-tasks.yandex/,0,NaN


In [8]:
cols = [f'CJ{i}' for i in range(1, 301)]

df = df[(df[cols] > 0).sum(axis=1) >= 15]

In [9]:
df

,status,invitation,relevance,lurker,recnum,code,testdata,itime,Q1,Q3a,...,date_9,date_10,date_11,Browser,BrowserVersion,OS,Device,Referer,Unsubscribed,Language
1,5,0,1,0,53,NaN,0,2026-04-28,1,5,...,28.04.2026 10:17:59,28.04.2026 10:18:15,NaN,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,Chrome 0.0,Win10,PC,https://iframe-tasks.yandex/,0,Russian
2,5,0,1,0,54,NaN,0,2026-04-28,1,5,...,28.04.2026 10:10:46,28.04.2026 10:11:05,NaN,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,Chrome 0.0,Win10,PC,https://iframe-tasks.yandex/,0,Russian
3,5,0,1,0,55,NaN,0,2026-04-28,1,5,...,28.04.2026 10:13:59,28.04.2026 10:14:12,NaN,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,Chrome 134.0,Win10,PC,https://iframe-tasks.yandex/,0,Russian
4,5,0,1,0,56,NaN,0,2026-04-28,1,5,...,28.04.2026 10:20:22,NaN,NaN,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,Chrome 0.0,Win10,PC,https://iframe-tasks.yandex/,0,Russian
6,5,0,1,0,58,NaN,0,2026-04-28,1,5,...,28.04.2026 10:24:10,28.04.2026 10:24:26,NaN,Mozilla/5.0 (iPhone; CPU iPhone OS 26_2_0 like...,Safari 0.0,iOS,Phone,https://iframe-tasks.yandex/,0,Russian
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1580,6,0,1,0,2069,NaN,0,2026-04-28,1,5,...,28.04.2026 18:05:39,28.04.2026 18:06:21,28.04.2026 18:06:39,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,Safari 0.0,MacOSX,PC,https://iframe-tasks.yandex/,0,NaN
1581,6,0,1,0,2070,NaN,0,2026-04-28,1,5,...,28.04.2026 17:59:11,28.04.2026 17:59:34,28.04.2026 18:00:00,Mozilla/5.0 (Linux; Android 10; K) AppleWebKit...,Chrome 0.0,Android,Phone,0,0,NaN
1582,5,0,1,0,2071,NaN,0,2026-04-28,1,5,...,28.04.2026 18:15:31,28.04.2026 18:15:53,NaN,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,Chrome 0.0,Win10,PC,https://iframe-tasks.yandex/,0,NaN
1583,6,0,1,0,2072,NaN,0,2026-04-28,1,5,...,28.04.2026 18:28:01,28.04.2026 18:28:39,28.04.2026 18:28:58,Mozilla/5.0 (Linux; arm_64; Android 13; 23053R...,Chrome 0.0,Android,Phone,https://iframe-tasks.yandex/,0,NaN


In [10]:
df = df[
    (df['Q3a'] == 5) &
    (df['Identityi'] == 3) &
    (df['Valenceh'] == 4)
]

In [11]:
df

,status,invitation,relevance,lurker,recnum,code,testdata,itime,Q1,Q3a,...,date_9,date_10,date_11,Browser,BrowserVersion,OS,Device,Referer,Unsubscribed,Language
1,5,0,1,0,53,NaN,0,2026-04-28,1,5,...,28.04.2026 10:17:59,28.04.2026 10:18:15,NaN,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,Chrome 0.0,Win10,PC,https://iframe-tasks.yandex/,0,Russian
2,5,0,1,0,54,NaN,0,2026-04-28,1,5,...,28.04.2026 10:10:46,28.04.2026 10:11:05,NaN,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,Chrome 0.0,Win10,PC,https://iframe-tasks.yandex/,0,Russian
4,5,0,1,0,56,NaN,0,2026-04-28,1,5,...,28.04.2026 10:20:22,NaN,NaN,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,Chrome 0.0,Win10,PC,https://iframe-tasks.yandex/,0,Russian
6,5,0,1,0,58,NaN,0,2026-04-28,1,5,...,28.04.2026 10:24:10,28.04.2026 10:24:26,NaN,Mozilla/5.0 (iPhone; CPU iPhone OS 26_2_0 like...,Safari 0.0,iOS,Phone,https://iframe-tasks.yandex/,0,Russian
7,6,0,1,0,59,NaN,0,2026-04-28,1,5,...,28.04.2026 10:23:49,28.04.2026 10:24:01,28.04.2026 10:24:06,Mozilla/5.0 (Linux; arm_64; Android 16; SM-S73...,Chrome 0.0,Android,Phone,https://iframe-tasks.yandex/,0,Russian
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1577,5,0,1,0,2066,NaN,0,2026-04-28,1,5,...,28.04.2026 17:38:51,28.04.2026 17:39:11,NaN,Mozilla/5.0 (Linux; Android 10; K) AppleWebKit...,Chrome 0.0,Android,Phone,https://iframe-tasks.yandex/,0,NaN
1578,5,0,1,0,2067,NaN,0,2026-04-28,1,5,...,28.04.2026 17:50:47,28.04.2026 17:51:14,NaN,Mozilla/5.0 (Linux; Android 10; K) AppleWebKit...,Chrome 0.0,Android,Phone,https://iframe-tasks.yandex/,0,NaN
1580,6,0,1,0,2069,NaN,0,2026-04-28,1,5,...,28.04.2026 18:05:39,28.04.2026 18:06:21,28.04.2026 18:06:39,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,Safari 0.0,MacOSX,PC,https://iframe-tasks.yandex/,0,NaN
1582,5,0,1,0,2071,NaN,0,2026-04-28,1,5,...,28.04.2026 18:15:31,28.04.2026 18:15:53,NaN,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,Chrome 0.0,Win10,PC,https://iframe-tasks.yandex/,0,NaN


In [12]:
df = df[df['Q44'] > 0]

In [13]:
df = df[df['Q5'] > 0]
df = df[df['Q6'] > 0]
df = df[df['Q41'] > 0]
df = df[df['Q42'] > 0]
df = df[df['Q43'] > 0]
df = df[df['Q44'] > 0]

In [14]:
df = df[df['Q5'] < 80]

In [15]:
df = df[df['Q5'] >= 18]

In [16]:
df.to_excel('conj_clean.xlsx', index=False)

In [17]:
df['Age_resp'] = df['Q5'].astype(int)
df = df[df['Q6'].isin([1, 2])]
df['Female_resp'] = (df['Q6'] == 2).astype(int)
df['Education_resp'] = df['Q42'].astype(int)
df['Income'] = df['Q43'].astype(int)
df['Locality'] = df['Q44'].astype(int)

In [18]:
mp = pd.read_excel('cj_mapping_final.xlsx')

In [19]:
df['Age_resp'].value_counts()

Age_resp
32    41
37    39
35    37
30    37
34    33
36    31
31    29
38    26
33    25
28    24
39    24
43    23
29    23
23    23
41    21
27    20
45    20
50    20
42    19
25    19
46    19
40    18
44    17
21    17
47    16
26    16
24    16
22    15
18    14
19    14
48    13
20    12
51    11
49    11
64    10
59     7
56     7
60     7
55     7
54     7
58     5
52     5
53     5
61     4
57     3
62     3
65     1
66     1
Name: count, dtype: int64

In [20]:
age_col = "Age_resp"
female_col = "Female_resp"
education_col = "Education_resp"
income_col = "Income"

rows = []

def add_panel(title):
    rows.append({"Переменная": title, "N": "", "Среднее": "", "Ст. откл.": "", "Мин.": "", "Макс.": ""})

def add_numeric(label, col):
    x = pd.to_numeric(df[col], errors="coerce")
    rows.append({
        "Переменная": label,
        "N": int(x.notna().sum()),
        "Среднее": round(x.mean(), 3),
        "Ст. откл.": round(x.std(), 3),
        "Мин.": round(x.min(), 3),
        "Макс.": round(x.max(), 3)
    })

def add_categorical(label_col):
    tab = df[label_col].value_counts(dropna=False).reset_index()
    tab.columns = ["category", "N"]
    tab["Доля"] = tab["N"] / tab["N"].sum()
    for _, r in tab.iterrows():
        rows.append({
            "Переменная": str(r["category"]),
            "N": int(r["N"]),
            "Среднее": round(r["Доля"], 3),
            "Ст. откл.": "",
            "Мин.": "",
            "Макс.": ""
        })

add_panel("Панель A: Описательные статистики")
add_numeric("Возраст", age_col)
add_numeric("Женщина (1 = да)", female_col)

add_panel("Панель B: Уровень образования")
add_categorical(education_col)

add_panel("Панель C: Уровень дохода")
add_categorical(income_col)

desc_table = pd.DataFrame(rows)
desc_table

,Переменная,N,Среднее,Ст. откл.,Мин.,Макс.
0,Панель A: Описательные статистики,,,,,
1,Возраст,815,36.46,10.59,18,66
2,Женщина (1 = да),815,0.463,0.499,0,1
3,Панель B: Уровень образования,,,,,
4,4.0,412,0.506,,,
5,2.0,231,0.283,,,
6,3.0,90,0.11,,,
7,1.0,82,0.101,,,
8,Панель C: Уровень дохода,,,,,
9,3.0,379,0.465,,,


In [21]:
latex_table = desc_table.to_latex(
    index=False,
    escape=True,
    caption="Описательная статистика и состав выборки",
    label="tab:desc_stats",
    column_format="lccccc",
    bold_rows=False
)

latex_table = latex_table.replace("\\toprule", "\\toprule")
latex_table = latex_table.replace("Панель A: Описательные статистики", "\\multicolumn{6}{l}{\\textit{Панель A: Описательные статистики}}")
latex_table = latex_table.replace("Панель B: Уровень образования", "\\multicolumn{6}{l}{\\textit{Панель B: Уровень образования}}")
latex_table = latex_table.replace("Панель C: Уровень дохода", "\\multicolumn{6}{l}{\\textit{Панель C: Уровень дохода}}")

with open("descriptive_statistics_sample.tex", "w", encoding="utf-8") as f:
    f.write(latex_table)

# Альфа-Кронбаха и средние

In [22]:
trust_cols = [col for col in df.columns if col.startswith("Trust")]
authoritar_cols = [col for col in df.columns if col.startswith("Authoritar")]
contact_cols = [col for col in df.columns if col.startswith("Contact")]
valence_cols = [col for col in df.columns if col.startswith("Valence")]

identity_civic_cols = ["Identitya", "Identityb", "Identityc", "Identityd"]
identity_ethnic_cols = ["Identitye", "Identityf", "Identityg", "Identityh"]

all_needed_cols = (
    identity_civic_cols + identity_ethnic_cols +
    trust_cols + authoritar_cols + contact_cols + valence_cols
)

df[all_needed_cols] = df[all_needed_cols].apply(pd.to_numeric, errors="coerce")

df[contact_cols] = df[contact_cols].replace(6, np.nan)
df[valence_cols] = df[valence_cols].replace(6, np.nan)

alpha_civic, ci_civic = pg.cronbach_alpha(data=df[identity_civic_cols])
alpha_ethnic, ci_ethnic = pg.cronbach_alpha(data=df[identity_ethnic_cols])
alpha_trust, ci_trust = pg.cronbach_alpha(data=df[trust_cols])
alpha_authoritar, ci_authoritar = pg.cronbach_alpha(data=df[authoritar_cols])
alpha_contact, ci_contact = pg.cronbach_alpha(data=df[contact_cols])
alpha_valence, ci_valence = pg.cronbach_alpha(data=df[valence_cols])

alpha_table = pd.DataFrame([
    ["Identity: civic", len(identity_civic_cols), round(alpha_civic, 3)],
    ["Identity: ethnic", len(identity_ethnic_cols), round(alpha_ethnic, 3)],
    ["Trust", len(trust_cols), round(alpha_trust, 3)],
    ["Authoritar", len(authoritar_cols), round(alpha_authoritar, 3)],
    ["Contact", len(contact_cols), round(alpha_contact, 3)],
    ["Valence", len(valence_cols), round(alpha_valence, 3)]
], columns=["Шкала", "Количество пунктов", "Альфа Кронбаха"])

alpha_table

,Шкала,Количество пунктов,Альфа Кронбаха
0,Identity: civic,4,0.772
1,Identity: ethnic,4,0.870
2,Trust,5,0.957
3,Authoritar,9,0.952
4,Contact,7,0.772
5,Valence,8,0.819


In [65]:
from pathlib import Path
alpha_table_ru = alpha_table.copy()

alpha_table_ru["Шкала"] = alpha_table_ru["Шкала"].replace({
    "Identity: civic": "Гражданская идентичность",
    "Identity: ethnic": "Этническая идентичность",
    "Trust": "Институциональное доверие",
    "Authoritar": "Авторитарные установки",
    "Contact": "Частота контакта с мигрантами",
    "Valence": "Валентность контакта с мигрантами"
})

Path("tables").mkdir(exist_ok=True)

alpha_table_ru.to_latex(
    "tables/cronbach_alpha_table.tex",
    index=False,
    caption="Надёжность шкал: альфа Кронбаха",
    label="tab:cronbach_alpha",
    column_format="lcc",
    float_format="%.3f",
    escape=False
)

In [23]:
moderators = {
    "Этническая идентичность": "identity_ethnic_index",
    "Гражданская идентичность": "identity_civic_index",
    "Институциональное доверие": "trust_index",
    "Авторитаризм": "authoritar_index",
    "Частота контакта": "contact_index",
    "Валентность контакта": "valence_index"
}

alphas = {
    "Этническая идентичность": 0.87,
    "Гражданская идентичность": 0.772,
    "Институциональное доверие": 0.957,
    "Авторитаризм": 0.952,
    "Частота контакта": 0.808,
    "Валентность контакта": 0.864
}

In [24]:
df["identity_civic_index"] = df[identity_civic_cols].mean(axis=1)
df["identity_ethnic_index"] = df[identity_ethnic_cols].mean(axis=1)
df["trust_index"] = df[trust_cols].mean(axis=1)
df["authoritar_index"] = df[authoritar_cols].mean(axis=1)
df["contact_index"] = df[contact_cols].mean(axis=1)
df["valence_index"] = df[valence_cols].mean(axis=1)

In [25]:
for name, col in moderators.items():
    x = pd.to_numeric(df[col], errors="coerce")
    print(f"{name} (α = {alphas[name]:.2f}, M = {x.mean():.2f}, SD = {x.std(ddof=1):.2f})")

Этническая идентичность (α = 0.87, M = 4.56, SD = 1.53)
Гражданская идентичность (α = 0.77, M = 5.81, SD = 0.99)
Институциональное доверие (α = 0.96, M = 3.05, SD = 1.19)
Авторитаризм (α = 0.95, M = 4.29, SD = 1.54)
Частота контакта (α = 0.81, M = 2.63, SD = 0.67)
Валентность контакта (α = 0.86, M = 3.44, SD = 0.53)


In [26]:
df["Supporta"] = pd.to_numeric(df["Supporta"], errors="coerce")
df["Supporta"] = df["Supporta"].replace(6, np.nan)

In [69]:
print(f"M = {df["Supporta"].mean():.2f}, SD = {df["Supporta"].std(ddof=1):.2f}")

M = 3.21, SD = 1.31


In [58]:
df["Supporta"].isna().sum()

21

In [27]:
df = df.reset_index(drop=True)
df["respondent_id"] = df.index + 1

In [28]:
cj_cols = [col for col in df.columns if col.startswith("CJ")]

In [29]:
index_cols = [
    "identity_civic_index",
    "identity_ethnic_index",
    "trust_index",
    "authoritar_index",
    "contact_index",
    "valence_index"
]

index_cols = [col for col in index_cols if col in df.columns]

# Перевод в long формат

In [30]:
controls = [col for col in df.columns if col not in cj_cols]

In [31]:
long_df = df.melt(
    id_vars=controls,
    value_vars=cj_cols,
    var_name="cj_var",
    value_name="pair_value"
)

In [32]:
long_df["pair_value"] = pd.to_numeric(long_df["pair_value"], errors="coerce")

In [33]:
long_df = long_df[long_df["pair_value"] > 0].copy()

In [34]:
long_df["cj_num"] = (long_df["cj_var"].str.replace("CJ", "", regex=False).astype(int))

In [35]:
long_df = long_df.sort_values(["respondent_id", "cj_num"]).copy()

long_df["task_num"] = (long_df.groupby("respondent_id").cumcount() + 1)

In [36]:
var_to_label = dict(zip(mp["variable_name"], mp["label"]))

In [37]:
long_df["cj_pair"] = long_df["cj_var"].map(var_to_label)

In [38]:
front_cols = ["respondent_id", "task_num", "cj_var", "cj_pair", "pair_value"]
other_cols = [col for col in long_df.columns if col not in front_cols]
long_df = long_df[front_cols + other_cols].reset_index(drop=True)

In [39]:
long_df.shape

(12225, 101)

In [40]:
choice_df = long_df.copy()

choice_df = choice_df.rename(columns={"pair_value": "choice"})

choice_df[["profile_1", "profile_2"]] = choice_df["cj_pair"].str.split("/", expand=True)

choice_df["profile_1"] = choice_df["profile_1"].str.strip()
choice_df["profile_2"] = choice_df["profile_2"].str.strip()

In [41]:
cand1 = choice_df.copy()

cand1["candidate"] = 1
cand1["profile"] = cand1["profile_1"]
cand1["outcome"] = (cand1["choice"] == 1).astype(int)

cand2 = choice_df.copy()

cand2["candidate"] = 2
cand2["profile"] = cand2["profile_2"]
cand2["outcome"] = (cand2["choice"] == 2).astype(int)

long_choice_df = pd.concat([cand1, cand2], ignore_index=True)
long_choice_df = long_choice_df.drop(columns=["profile_1", "profile_2"])

In [42]:
front_cols = ["respondent_id","task_num","cj_var", "candidate", "profile", "choice", "outcome"]

other_cols = [col for col in long_choice_df.columns if col not in front_cols]

long_choice_df = long_choice_df[front_cols + other_cols]

long_choice_df = long_choice_df.sort_values(["respondent_id", "task_num", "candidate"]).reset_index(drop=True)

long_choice_df

,respondent_id,task_num,cj_var,candidate,profile,choice,outcome,cj_pair,status,invitation,...,Education_resp,Income,Locality,identity_civic_index,identity_ethnic_index,trust_index,authoritar_index,contact_index,valence_index,cj_num
0,1,1,CJ51,1,R2M1Ed3Em1G1A4O2L2P3Ap2,1,1,R2M1Ed3Em1G1A4O2L2P3Ap2 / R3M3Ed2Em2G2A4O2L1P4Ap2,5,0,...,3,3,2,6.25,6.75,1.8,5.000000,3.000000,3.25,51
1,1,1,CJ51,2,R3M3Ed2Em2G2A4O2L1P4Ap2,1,0,R2M1Ed3Em1G1A4O2L2P3Ap2 / R3M3Ed2Em2G2A4O2L1P4Ap2,5,0,...,3,3,2,6.25,6.75,1.8,5.000000,3.000000,3.25,51
2,1,2,CJ99,1,R1M3Ed1Em1G2A1O2L2P1Ap2,2,0,R1M3Ed1Em1G2A1O2L2P1Ap2 / R6M3Ed3Em2G1A7O3L2P1Ap1,5,0,...,3,3,2,6.25,6.75,1.8,5.000000,3.000000,3.25,99
3,1,2,CJ99,2,R6M3Ed3Em2G1A7O3L2P1Ap1,2,1,R1M3Ed1Em1G2A1O2L2P1Ap2 / R6M3Ed3Em2G1A7O3L2P1Ap1,5,0,...,3,3,2,6.25,6.75,1.8,5.000000,3.000000,3.25,99
4,1,3,CJ118,1,R2M4Ed3Em2G1A6O3L1P2Ap2,2,0,R2M4Ed3Em2G1A6O3L1P2Ap2 / R3M4Ed3Em2G2A6O4L1P4Ap1,5,0,...,3,3,2,6.25,6.75,1.8,5.000000,3.000000,3.25,118
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24445,815,13,CJ239,2,R7M1Ed2Em2G2A2O4L1P1Ap2,1,0,R2M1Ed1Em1G2A2O4L2P2Ap2 / R7M1Ed2Em2G2A2O4L1P1Ap2,6,0,...,2,3,3,5.75,4.75,3.0,4.444444,3.857143,3.75,239
24446,815,14,CJ246,1,R1M1Ed2Em2G2A2O3L1P4Ap1,2,0,R1M1Ed2Em2G2A2O3L1P4Ap1 / R1M1Ed1Em2G2A2O3L1P2Ap2,6,0,...,2,3,3,5.75,4.75,3.0,4.444444,3.857143,3.75,246
24447,815,14,CJ246,2,R1M1Ed1Em2G2A2O3L1P2Ap2,2,1,R1M1Ed2Em2G2A2O3L1P4Ap1 / R1M1Ed1Em2G2A2O3L1P2Ap2,6,0,...,2,3,3,5.75,4.75,3.0,4.444444,3.857143,3.75,246
24448,815,15,CJ266,1,R2M2Ed2Em1G1A1O3L1P3Ap1,1,1,R2M2Ed2Em1G1A1O3L1P3Ap1 / R2M2Ed2Em1G1A9O4L2P2Ap1,6,0,...,2,3,3,5.75,4.75,3.0,4.444444,3.857143,3.75,266


In [43]:
long_choice_df.groupby(["respondent_id", "task_num"]).size().value_counts()

2    12225
Name: count, dtype: int64

In [44]:
long_choice_df.groupby(["respondent_id", "task_num"])["outcome"].sum().value_counts()

outcome
1    12225
Name: count, dtype: int64

In [45]:
long_choice_df.groupby("respondent_id")["task_num"].nunique().describe()

count    815.0
mean      15.0
std        0.0
min       15.0
25%       15.0
50%       15.0
75%       15.0
max       15.0
Name: task_num, dtype: float64

In [46]:
attrs = ["region","motivation","education","employer","gender","age","occupation","language","politics","appearance"]
profile_map_A = pd.DataFrame()
profile_map_A["profile"] = mp["label"].str.split("/", expand=True)[0].str.strip()

for attr in attrs:
    profile_map_A[attr] = mp["A_" + attr]
profile_map_B = pd.DataFrame()
profile_map_B["profile"] = mp["label"].str.split("/", expand=True)[1].str.strip()

for attr in attrs:
    profile_map_B[attr] = mp["B_" + attr]
profile_map = pd.concat([profile_map_A, profile_map_B], ignore_index=True)
profile_map = profile_map.drop_duplicates(subset=["profile"])
long_choice_df = long_choice_df.merge(
    profile_map,
    on="profile",
    how="left",
    validate="m:1"
)

long_choice_df

,respondent_id,task_num,cj_var,candidate,profile,choice,outcome,cj_pair,status,invitation,...,region,motivation,education,employer,gender,age,occupation,language,politics,appearance
0,1,1,CJ51,1,R2M1Ed3Em1G1A4O2L2P3Ap2,1,1,R2M1Ed3Em1G1A4O2L2P3Ap2 / R3M3Ed2Em2G2A4O2L1P4Ap2,5,0,...,"Восточная Азия (Китай, Монголия, Южная Корея)",Бегство от вооружённого конфликта,Среднее специальное (колледж),Государственная организация,Женщина,45,Программист,Говорит свободно,Страна помогает России обходить санкции (парал...,В повседневной жизни носит традиционную национ...
1,1,1,CJ51,2,R3M3Ed2Em2G2A4O2L1P4Ap2,1,0,R2M1Ed3Em1G1A4O2L2P3Ap2 / R3M3Ed2Em2G2A4O2L1P4Ap2,5,0,...,"Восточная Европа (Беларусь, Молдова, Украина)","Воссоединение с супругом(ой), ранее приехавшим...",Среднее (школа),Небольшая частная компания,Мужчина,45,Программист,Говорит плохо,Страна присоединилась к санкциям против России,В повседневной жизни носит традиционную национ...
2,1,2,CJ99,1,R1M3Ed1Em1G2A1O2L2P1Ap2,2,0,R1M3Ed1Em1G2A1O2L2P1Ap2 / R6M3Ed3Em2G1A7O3L2P1Ap1,5,0,...,"Африка (Кения, Нигерия, ЮАР)","Воссоединение с супругом(ой), ранее приехавшим...",Высшее (университет),Государственная организация,Мужчина,21,Программист,Говорит свободно,Страна обычно голосует вместе с Россией в Гена...,В повседневной жизни носит традиционную национ...
3,1,2,CJ99,2,R6M3Ed3Em2G1A7O3L2P1Ap1,2,1,R1M3Ed1Em1G2A1O2L2P1Ap2 / R6M3Ed3Em2G1A7O3L2P1Ap1,5,0,...,"Юго-Восточная Европа (Болгария, Венгрия, Сербия)","Воссоединение с супругом(ой), ранее приехавшим...",Среднее специальное (колледж),Небольшая частная компания,Женщина,61,Строитель,Говорит свободно,Страна обычно голосует вместе с Россией в Гена...,В повседневной жизни носит обычную светскую од...
4,1,3,CJ118,1,R2M4Ed3Em2G1A6O3L1P2Ap2,2,0,R2M4Ed3Em2G1A6O3L1P2Ap2 / R3M4Ed3Em2G2A6O4L1P4Ap1,5,0,...,"Восточная Азия (Китай, Монголия, Южная Корея)",Поиск работы,Среднее специальное (колледж),Небольшая частная компания,Женщина,47,Строитель,Говорит плохо,Страна обычно голосует вместе с США в Генассам...,В повседневной жизни носит традиционную национ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24445,815,13,CJ239,2,R7M1Ed2Em2G2A2O4L1P1Ap2,1,0,R2M1Ed1Em1G2A2O4L2P2Ap2 / R7M1Ed2Em2G2A2O4L1P1Ap2,6,0,...,"Южный Кавказ (Азербайджан, Армения, Грузия)",Бегство от вооружённого конфликта,Среднее (школа),Небольшая частная компания,Мужчина,22,Сфера услуг (общепит),Говорит плохо,Страна обычно голосует вместе с Россией в Гена...,В повседневной жизни носит традиционную национ...
24446,815,14,CJ246,1,R1M1Ed2Em2G2A2O3L1P4Ap1,2,0,R1M1Ed2Em2G2A2O3L1P4Ap1 / R1M1Ed1Em2G2A2O3L1P2Ap2,6,0,...,"Африка (Кения, Нигерия, ЮАР)",Бегство от вооружённого конфликта,Среднее (школа),Небольшая частная компания,Мужчина,22,Строитель,Говорит плохо,Страна присоединилась к санкциям против России,В повседневной жизни носит обычную светскую од...
24447,815,14,CJ246,2,R1M1Ed1Em2G2A2O3L1P2Ap2,2,1,R1M1Ed2Em2G2A2O3L1P4Ap1 / R1M1Ed1Em2G2A2O3L1P2Ap2,6,0,...,"Африка (Кения, Нигерия, ЮАР)",Бегство от вооружённого конфликта,Высшее (университет),Небольшая частная компания,Мужчина,22,Строитель,Говорит плохо,Страна обычно голосует вместе с США в Генассам...,В повседневной жизни носит традиционную национ...
24448,815,15,CJ266,1,R2M2Ed2Em1G1A1O3L1P3Ap1,1,1,R2M2Ed2Em1G1A1O3L1P3Ap1 / R2M2Ed2Em1G1A9O4L2P2Ap1,6,0,...,"Восточная Азия (Китай, Монголия, Южная Корея)",Бегство от политических преследований,Среднее (школа),Государственная организация,Женщина,21,Строитель,Говорит плохо,Страна помогает России обходить санкции (парал...,В повседневной жизни носит обычную светскую од...


In [47]:
reg_df = long_choice_df.copy()
reg_df["outcome"] = pd.to_numeric(reg_df["outcome"], errors="coerce")
reg_df["choice"] = pd.to_numeric(reg_df["choice"], errors="coerce")
reg_df = reg_df.dropna(subset=["outcome"]).copy()

reg_df["outcome"] = reg_df["outcome"].astype(int)

In [48]:
attr_cols = ["region","motivation", "education", "employer", "gender", "age", "occupation", "language", "politics", "appearance"]

attr_cols = [col for col in attr_cols if col in reg_df.columns]

for col in attr_cols:
    reg_df[col] = reg_df[col].astype("category")

In [49]:
index_cols = ["identity_civic_index","identity_ethnic_index", "trust_index", "authoritar_index", "contact_index", "valence_index"]

index_cols = [col for col in index_cols if col in reg_df.columns]

for col in index_cols:
    reg_df[col] = pd.to_numeric(reg_df[col], errors="coerce")

In [50]:
attr_cols = ["region", "motivation", "education", "employer", "gender", "age", "occupation", "language", "politics", "appearance"]

attr_cols = [col for col in attr_cols if col in reg_df.columns]

for col in attr_cols:
    reg_df[col] = reg_df[col].astype("category")

In [51]:
for col in index_cols:
    reg_df[col + "_c"] = reg_df[col] - reg_df[col].mean()

In [52]:
reg_df[
    [
        "respondent_id",
        "task_num",
        "candidate",
        "profile",
        "choice",
        "outcome",
        "region",
        "motivation",
        "education",
        "employer",
        "gender",
        "age",
        "occupation",
        "language",
        "politics",
        "appearance",
        "identity_civic_index",
        "identity_ethnic_index",
        "trust_index",
        "authoritar_index",
        "contact_index",
        "valence_index", 
        "Age_resp", 
        "Female_resp",
        "Income", 
        "Locality",
        "Education_resp", 
        "Supporta"
    ]
].head(10)

,respondent_id,task_num,candidate,profile,choice,outcome,region,motivation,education,employer,...,trust_index,authoritar_index,contact_index,valence_index,Age_resp,Female_resp,Income,Locality,Education_resp,Supporta
0,1,1,1,R2M1Ed3Em1G1A4O2L2P3Ap2,1,1,"Восточная Азия (Китай, Монголия, Южная Корея)",Бегство от вооружённого конфликта,Среднее специальное (колледж),Государственная организация,...,1.8,5.0,3.0,3.25,44,1,3,2,3,1.0
1,1,1,2,R3M3Ed2Em2G2A4O2L1P4Ap2,1,0,"Восточная Европа (Беларусь, Молдова, Украина)","Воссоединение с супругом(ой), ранее приехавшим...",Среднее (школа),Небольшая частная компания,...,1.8,5.0,3.0,3.25,44,1,3,2,3,1.0
2,1,2,1,R1M3Ed1Em1G2A1O2L2P1Ap2,2,0,"Африка (Кения, Нигерия, ЮАР)","Воссоединение с супругом(ой), ранее приехавшим...",Высшее (университет),Государственная организация,...,1.8,5.0,3.0,3.25,44,1,3,2,3,1.0
3,1,2,2,R6M3Ed3Em2G1A7O3L2P1Ap1,2,1,"Юго-Восточная Европа (Болгария, Венгрия, Сербия)","Воссоединение с супругом(ой), ранее приехавшим...",Среднее специальное (колледж),Небольшая частная компания,...,1.8,5.0,3.0,3.25,44,1,3,2,3,1.0
4,1,3,1,R2M4Ed3Em2G1A6O3L1P2Ap2,2,0,"Восточная Азия (Китай, Монголия, Южная Корея)",Поиск работы,Среднее специальное (колледж),Небольшая частная компания,...,1.8,5.0,3.0,3.25,44,1,3,2,3,1.0
5,1,3,2,R3M4Ed3Em2G2A6O4L1P4Ap1,2,1,"Восточная Европа (Беларусь, Молдова, Украина)",Поиск работы,Среднее специальное (колледж),Небольшая частная компания,...,1.8,5.0,3.0,3.25,44,1,3,2,3,1.0
6,1,4,1,R2M4Ed1Em2G2A2O4L1P2Ap2,2,0,"Восточная Азия (Китай, Монголия, Южная Корея)",Поиск работы,Высшее (университет),Небольшая частная компания,...,1.8,5.0,3.0,3.25,44,1,3,2,3,1.0
7,1,4,2,R6M3Ed1Em2G2A8O4L1P1Ap2,2,1,"Юго-Восточная Европа (Болгария, Венгрия, Сербия)","Воссоединение с супругом(ой), ранее приехавшим...",Высшее (университет),Небольшая частная компания,...,1.8,5.0,3.0,3.25,44,1,3,2,3,1.0
8,1,5,1,R6M1Ed3Em1G2A6O2L1P1Ap1,1,1,"Юго-Восточная Европа (Болгария, Венгрия, Сербия)",Бегство от вооружённого конфликта,Среднее специальное (колледж),Государственная организация,...,1.8,5.0,3.0,3.25,44,1,3,2,3,1.0
9,1,5,2,R6M1Ed3Em1G1A1O2L2P1Ap2,1,0,"Юго-Восточная Европа (Болгария, Венгрия, Сербия)",Бегство от вооружённого конфликта,Среднее специальное (колледж),Государственная организация,...,1.8,5.0,3.0,3.25,44,1,3,2,3,1.0


In [53]:
reg_df["age"] = np.select([reg_df["age"].isin([21, 22, 23]), reg_df["age"].isin([45, 46, 47]), reg_df["age"].isin([61, 62, 63])],
    ["20","45","60"],default=np.nan)

reg_df["age"].value_counts(dropna=False)

age
45    8419
60    8287
20    7744
Name: count, dtype: int64

In [54]:
reg_df.to_excel('long_df_SA.xlsx', index=False)